# 07 Top Genes
Checks which genes the logistic regression model relies on most for each antibiotic and whether these are known resistance genes. Known resistance genes come from the ResFinder results in the PanPred repository.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

src_dir = Path("../src").resolve()
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
import data

X, labels, year, clusters = data.load_data()
main_antibiotics = ["CIP", "GEN", "CXM"]

## Resistance gene names in X

First, I checked whether known resistance genes can be found in X by name, by searching the gene names for typical resistance gene prefixes.

In [4]:
for pattern in ["bla", "aac", "aad", "aph", "dfr", "sul", "qnr", "tet"]:
    matches = [name for name in X.columns if pattern in name.lower()]
    print(pattern, len(matches), matches[:5])

bla 4 ['blaSE', 'bla', 'bla_1', 'bla_3']
aac 3 ['paaC', 'aac3-VI', 'aacA-aphD']
aad 3 ['aadA', 'aadA1', 'aadA_1']
aph 3 ['papH', 'dapH', 'aacA-aphD']
dfr 3 ['ydfR', 'dfrD', 'dfrA']
sul 2 ['sulA', 'sulI']
qnr 0 []
tet 5 ['tetA', 'tetR', 'tetC', 'tetD', 'tetR_2']


Some resistance genes appear under readable names (for example aadA and dfrA), but the names are not reliable: some are vague (bla, bla_1), some are spelled differently from ResFinder (aac3-VI instead of aac(3)-VI), and some are unrelated genes that only contain the pattern (sulA, papH). Many genes in X are also only named group_ with a number. Because of this, I match genes by which strains carry them instead of by name.

## Known resistance genes (ResFinder)

The ResFinder_resistance file lists, for every strain, the known acquired resistance genes that ResFinder found in its genome. PanPred also has a CARD_resistance file, but it was not used: it is based on the ARG-ANNOT database and includes genes that every E. coli has (like AmpC), which cannot explain differences in resistance.

I turn the file into a 0/1 table of strains × known genes in four steps:
1. Read the file, keeping only the strain and gene columns.
2. Remove the header lines that are repeated before each strain's block in the file.
3. Mark each gene as present (1) or absent (0) for each strain. Several versions of the same gene count as one.
4. Put the strains in the same order as X, with 0 for strains where no known gene was found.

In [7]:
known = pd.read_csv(
    "../data/PanPred/ResFinder_resistance",
    sep="\t",
    usecols=["Sample", "gene"],
    on_bad_lines="warn",
)

In [8]:
known = known[known["Sample"] != "Sample"]

In [9]:
known_genes = (pd.crosstab(known["Sample"], known["gene"]) > 0).astype(int)

In [10]:
known_genes = known_genes.reindex(X.index, fill_value=0)

In [11]:
print(known_genes.shape)
print("Strains with at least one known gene:", (known_genes.sum(axis=1) > 0).sum())
known_genes.sum().sort_values(ascending=False).head(15)

(1936, 67)
Strains with at least one known gene: 1361


gene
blaTEM       934
dfrA         782
aadA         711
sul2         611
sul1         595
strA         588
strB         578
tet(A)       435
mph(A)       402
blaCTX-M     351
tet(B)       329
cat          308
aac(3)-II    252
blaOXA       187
aac(6')      164
dtype: int64

### Interpretation

The known_genes table has 67 known resistance genes and includes all 1936 strains. 1361 strains (about 70%) carry at least one known resistance gene.

The most common genes are blaTEM (β-lactamase, 934 strains), dfrA (trimethoprim, 782) and aadA (aminoglycosides, 711).

Two genes relate directly to the main antibiotics. aac(3)-II, which gives resistance to gentamicin, is found in 252 strains, close to the 284 GEN-resistant strains, so one known gene could explain most GEN resistance. blaCTX-M is a β-lactamase that breaks down cephalosporins and is found in 351 strains, fewer than the 520 CXM-resistant strains, so CXM resistance likely has other causes as well.

## Quinolone resistance genes check
Quinolone resistance genes are a known gene-based route to gain CIP resistance. The check below is o see whether resistance to CIP can potentially be explained by these genes.

In [14]:
[gene for gene in known_genes.columns if "qnr" in gene.lower() or "oqx" in gene.lower()]

['Qnr-A', 'Qnr-B', 'Qnr-D', 'Qnr-S', 'oqxA', 'oqxB']

In [15]:
quinolone_genes = ["Qnr-A", "Qnr-B", "Qnr-D", "Qnr-S", "oqxA", "oqxB"]
print(known_genes[quinolone_genes].sum())

has_quinolone_gene = known_genes[quinolone_genes].sum(axis=1) > 0
print("Strains with at least one:", has_quinolone_gene.sum())

gene
Qnr-A     1
Qnr-B     5
Qnr-D     2
Qnr-S    93
oqxA      1
oqxB      1
dtype: int64
Strains with at least one: 100


In [16]:
cip = labels["CIP"]
is_resistant = cip == "R"

print("CIP-resistant strains:", is_resistant.sum())
print("…of which carry a quinolone gene:", (is_resistant & has_quinolone_gene).sum())
print("Quinolone-gene carriers that are CIP-resistant:", (is_resistant & has_quinolone_gene).sum(), "of", has_quinolone_gene.sum())

CIP-resistant strains: 429
…of which carry a quinolone gene: 35
Quinolone-gene carriers that are CIP-resistant: 35 of 100


### Interpretation

Acquired quinolone resistance genes (Qnr-A, Qnr-B, Qnr-D, Qnr-S, oqxA, oqxB) are found in 100 strains. Only 35 of these are CIP-resistant, which fits with these genes giving only weak resistance on their own. Of the 429 CIP-resistant strains, only 35 (8%) carry one of these genes. So most CIP resistance in this dataset cannot be explained by known genes and likely comes from mutations in which neither X nor ResFinder can see. This supports the hypothesis.


## Logistic Regression Model for Top Genes

A logistic regression model is trained per antibiotic on all labelled strains, with the same settings as in notebook 05. There is no test dataset: the objective is to look inside the model, not to score it, and training on all strains gives the most stable weights.

After training, the model has one weight per gene:

- weight > 0: the gene pushes the prediction towards resistant
- weight ≈ 0: the gene does not really influence the prediction
- weight < 0: the gene pushes the prediction towards susceptible

The 20 genes with the largest positive weights are treated as the strongest signs of resistance. For each of them, the table also shows how many strains carry the gene (n_strains) and what percentage of these carriers are resistant (pct_R).

Genes that are always found in the same strains share their weight, so the exact ranking is not precise. The overall pattern of the top 20 matters more than the position of a single gene.

In [19]:
top_genes = {}

for antibiotic in main_antibiotics:
    y = (labels[antibiotic].dropna() == "R").astype(int)
    X_labelled = X.loc[y.index]

    model = LogisticRegression(class_weight="balanced", max_iter=1000)
    model.fit(X_labelled, y)

    weights = pd.Series(model.coef_[0], index=X.columns)
    top = weights.sort_values(ascending=False).head(20).to_frame("weight")
    
    carriers = X_labelled[top.index]
    top["n_strains"] = carriers.sum()
    top["pct_R"] = (carriers.mul(y, axis=0).sum() / carriers.sum() * 100).round(1)

    top_genes[antibiotic] = top
    print(antibiotic, "done")

CIP done
GEN done
CXM done


In [20]:
top_genes["GEN"]

,weight,n_strains,pct_R
aac3-VI,2.836975,232,97.8
tmrB,1.575730,163,96.9
group_17589,1.333757,75,100.0
group_2588,1.023042,112,98.2
group_17191,1.008045,113,98.2
aadA_1,0.835603,18,83.3
aadA1,0.759397,328,24.1
cat,0.600058,424,45.0
group_326,0.588888,395,16.2
group_3662,0.556028,47,95.7


### GEN

The top gene is aac3-VI, an aac(3) gene that inactivates gentamicin. This is the known resistance mechanism for GEN. Its weight (2.84) is almost twice as high as the next gene, and 97.8% of the 232 strains that carry it are resistant.

Most of the other top genes are either other resistance genes (aadA, cat, bla_1), genes of mobile DNA elements that carry resistance genes between bacteria (tnsB, tniBdelta1), or unnamed genes that are almost always found in resistant strains and likely travel together with the resistance gene. Almost none of them look like family markers.

This fits with GEN's small drop under the lineage split: the model mostly relies on the real resistance gene, which works in any family.

In [22]:
top_genes["CXM"]

,weight,n_strains,pct_R
group_17190,2.046521,300,96.0
bla_1,1.607593,239,99.6
wbuC,1.596974,233,99.6
aacA-aphD,0.714112,149,91.3
group_4021,0.658795,249,29.3
aac3-VI,0.589550,232,82.3
glpR_6,0.577394,48,27.1
group_2565,0.545504,1482,25.4
folP2,0.533378,1181,37.6
group_2543,0.533182,86,53.5


### CXM

The top genes are group_17190, bla_1 and wbuC. bla_1 is a β-lactamase, which breaks down drugs like cefuroxime, so it is the known resistance mechanism. Almost all carriers of these three genes are resistant (96–99.6%). bla_1 and wbuC are found in almost the same strains, so they are likely carried together.

After these, the picture changes. Most of the remaining genes are very common (for example adiC in 1891 strains) and their carriers are resistant at about the same rate as all strains (around 27%). They do not look like resistance genes and may be family markers. group_28605 is found in only 2 strains, which is too few to mean anything.

Compared with GEN, the model relies less on clear resistance genes. This fits with CXM being in between: part of its resistance comes from a gene the model can see and part from causes it cannot see.

In [24]:
top_genes["CIP"]

,weight,n_strains,pct_R
sopA,0.505345,825,44.0
dicB,0.490813,867,24.9
folP2,0.478552,1181,34.3
tnpA,0.458008,751,44.3
aac3-VI,0.437201,232,65.5
group_1811,0.405888,719,31.8
group_11597,0.404272,231,31.6
sopB,0.403350,800,44.2
cbl,0.372701,1289,20.6
yfjJ,0.371804,657,13.7


### CIP

The CIP table looks very different from GEN and CXM. No gene stands out: the weights are small and close together (0.33–0.51), compared with 2.84 for the top GEN gene. None of the top 20 genes is a known quinolone resistance gene.

Most of the top genes are very common (650–1500 strains), and their carriers are resistant at only 14–50%, not far from the overall CIP rate (22%). Many are genes of plasmids, mobile DNA or viruses that infect bacteria (sopA, sopB, tnpA, insD_2, dicB), which are typical of certain families. The few resistance genes in the list (aac3-VI, bla_1, aacA-aphD) give resistance to other antibiotics, not to CIP, and likely appear because CIP-resistant strains often belong to multi-resistant lineages.

So for CIP, the model has no real resistance gene to use, and spreads its weight over many genes that mark families. This fits with CIP's large drop under the lineage split, and supports the hypothesis.

## Matching top genes with known resistance genes

Gene names in X are not reliable, so I check the top genes by pattern instead: if a top gene is found in almost exactly the same strains as a known ResFinder gene, it is very likely the same gene or carried together with it.

For each of the top 20 genes, I compare it with all 67 known genes using the Jaccard similarity: of the strains that have either gene, the share that have both. 1 means the two genes are found in exactly the same strains; 0 means they never occur together. I keep the known gene with the highest similarity and count it as a match if the similarity is at least 0.9.

A match only shows that two genes are found in the same strains. It can mean the same gene under a different name or a different gene that is always carried together with it, for example, on the same plasmid.

In [27]:
def jaccard(a, b):
    return (a & b).sum() / (a | b).sum()


In [28]:
def best_known_match(gene):
    gene_column = X[gene].astype(bool)
    best_name, best_score = None, 0.0
    for known in known_genes.columns:
        score = jaccard(gene_column, known_genes[known].astype(bool))
        if score > best_score:
            best_name, best_score = known, score
    return best_name, best_score

In [29]:
for antibiotic in main_antibiotics:
    matches = [best_known_match(gene) for gene in top_genes[antibiotic].index]
    top_genes[antibiotic]["best_known_match"] = [name for name, score in matches]
    top_genes[antibiotic]["jaccard"] = [round(score, 2) for name, score in matches]
    top_genes[antibiotic]["is_match"] = top_genes[antibiotic]["jaccard"] >= 0.9

In [30]:
pd.Series({antibiotic: top_genes[antibiotic]["is_match"].sum() for antibiotic in main_antibiotics},
          name="top 20 genes matching a known resistance gene")

CIP    2
GEN    4
CXM    3
Name: top 20 genes matching a known resistance gene, dtype: int64

In [31]:
top_genes["GEN"]

,weight,n_strains,pct_R,best_known_match,jaccard,is_match
aac3-VI,2.836975,232,97.8,aac(3)-II,0.92,True
tmrB,1.575730,163,96.9,aac(3)-II,0.65,False
group_17589,1.333757,75,100.0,aac(3)-II,0.30,False
group_2588,1.023042,112,98.2,aac(3)-II,0.44,False
group_17191,1.008045,113,98.2,aac(3)-II,0.45,False
aadA_1,0.835603,18,83.3,aadB,1.00,True
aadA1,0.759397,328,24.1,aadA,0.46,False
cat,0.600058,424,45.0,mph(A),0.64,False
group_326,0.588888,395,16.2,blaTEM,0.21,False
group_3662,0.556028,47,95.7,aac(3)-II,0.19,False


In [32]:
top_genes["CXM"]

,weight,n_strains,pct_R,best_known_match,jaccard,is_match
group_17190,2.046521,300,96.0,blaCTX-M,0.74,False
bla_1,1.607593,239,99.6,blaCTX-M,0.68,False
wbuC,1.596974,233,99.6,blaCTX-M,0.66,False
aacA-aphD,0.714112,149,91.3,aac(6'),0.91,True
group_4021,0.658795,249,29.3,sul1,0.14,False
aac3-VI,0.589550,232,82.3,aac(3)-II,0.92,True
glpR_6,0.577394,48,27.1,blaLEN,0.91,True
group_2565,0.545504,1482,25.4,blaTEM,0.45,False
folP2,0.533378,1181,37.6,dfrA,0.54,False
group_2543,0.533182,86,53.5,blaOXA,0.08,False


In [33]:
top_genes["CIP"]

,weight,n_strains,pct_R,best_known_match,jaccard,is_match
sopA,0.505345,825,44.0,dfrA,0.43,False
dicB,0.490813,867,24.9,blaTEM,0.30,False
folP2,0.478552,1181,34.3,dfrA,0.54,False
tnpA,0.458008,751,44.3,dfrA,0.43,False
aac3-VI,0.437201,232,65.5,aac(3)-II,0.92,True
group_1811,0.405888,719,31.8,blaTEM,0.32,False
group_11597,0.404272,231,31.6,dfrA,0.16,False
sopB,0.403350,800,44.2,dfrA,0.43,False
cbl,0.372701,1289,20.6,blaTEM,0.38,False
yfjJ,0.371804,657,13.7,blaTEM,0.23,False


### Interpretation

GEN has the most matches (4), followed by CXM (3) and CIP (2). More important is which genes match. For GEN, the top gene aac3-VI matches aac(3)-II (similarity 0.92), confirming that the model's strongest gene is the known gentamicin resistance gene. Two other matches (aadB and aac(6')) are also aminoglycoside resistance genes.

For CXM, none of the matches is a clear cefuroxime resistance gene. However, the three top genes all come closest to blaCTX-M (0.66–0.74), a β-lactamase that breaks down cephalosporins. Their similarity stays below 0.9 probably because they are found in only part of the strains carrying blaCTX-M.

For CIP, both matches are aminoglycoside resistance genes, which give resistance to other antibiotics. No top CIP gene matches a quinolone resistance gene. This supports the hypothesis: the CIP model does not rely on a resistance gene, but on genes that are common in resistant families.

The 0.9 threshold is strict. It misses genes that are carried together with a resistance gene in only some strains, so the number of matches is a lower limit.